In [19]:
from google.colab import files

uploaded = files.upload()

Saving Online Retail.xlsx to Online Retail (1).xlsx


In [20]:
import pandas as pd

df = pd.read_excel("/content/Online Retail.xlsx")

print("Dataset shape:", df.shape)
display(df.head(10))

Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [21]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nQuantity range:")
print("Min:", df["Quantity"].min())
print("Max:", df["Quantity"].max())

print("\nUnitPrice range:")
print("Min:", df["UnitPrice"].min())
print("Max:", df["UnitPrice"].max())

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows:
5268

Quantity range:
Min: -80995
Max: 80995

UnitPrice range:
Min: -11062.06
Max: 38970.0


In [22]:
negative_quantity = df[df["Quantity"] < 0]

print("Rows with negative Quantity:", len(negative_quantity))

display(
    negative_quantity[
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
    ].head(10)
)

Rows with negative Quantity: 10624


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
141,C536379,D,Discount,-1,27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65
939,C536506,22960,JAM MAKING SET WITH JARS,-6,4.25


In [23]:
negative_price = df[df["UnitPrice"] < 0]

print("Rows with negative UnitPrice:", len(negative_price))

display(
    negative_price[
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "Country"]
    ]
)

Rows with negative UnitPrice: 2


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,Country
299983,A563186,B,Adjust bad debt,1,-11062.06,United Kingdom
299984,A563187,B,Adjust bad debt,1,-11062.06,United Kingdom


In [24]:
!pip install -q pyspark==4.0.4 delta-spark==4.0.0 openpyxl

In [25]:
import pandas as pd

excel_df = pd.read_excel("/content/Online Retail.xlsx")

csv_path = "/content/online_retail.csv"
excel_df.to_csv(csv_path, index=False)

print("CSV file created successfully!")
print("Path:", csv_path)

CSV file created successfully!
Path: /content/online_retail.csv


In [26]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("EcommerceSalesDataPipeline")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark + Delta Lake started successfully!")
print("Spark version:", spark.version)

Spark + Delta Lake started successfully!
Spark version: 4.0.4


In [27]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/content/online_retail.csv")
)

print("Data successfully ingested into Spark!")
print("Number of rows:", raw_df.count())

raw_df.printSchema()
raw_df.show(5, truncate=False)

Data successfully ingested into Spark!
Number of rows: 541909
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850.0   

In [28]:
from pyspark.sql.functions import col, trim, count

quality_df = (
    raw_df

    # 1. Completeness Check - Description must exist
    .withColumn(
        "description_valid",
        col("Description").isNotNull() & (trim(col("Description")) != "")
    )

    # 2. Completeness Check - CustomerID must exist
    .withColumn(
        "customer_valid",
        col("CustomerID").isNotNull()
    )

    # 3. Accuracy / Business Rule - UnitPrice must be positive
    .withColumn(
        "price_valid",
        col("UnitPrice") > 0
    )
)

quality_df.select(
    "InvoiceNo",
    "Description",
    "CustomerID",
    "UnitPrice",
    "description_valid",
    "customer_valid",
    "price_valid"
).show(10, truncate=False)

+---------+-----------------------------------+----------+---------+-----------------+--------------+-----------+
|InvoiceNo|Description                        |CustomerID|UnitPrice|description_valid|customer_valid|price_valid|
+---------+-----------------------------------+----------+---------+-----------------+--------------+-----------+
|536365   |WHITE HANGING HEART T-LIGHT HOLDER |17850.0   |2.55     |true             |true          |true       |
|536365   |WHITE METAL LANTERN                |17850.0   |3.39     |true             |true          |true       |
|536365   |CREAM CUPID HEARTS COAT HANGER     |17850.0   |2.75     |true             |true          |true       |
|536365   |KNITTED UNION FLAG HOT WATER BOTTLE|17850.0   |3.39     |true             |true          |true       |
|536365   |RED WOOLLY HOTTIE WHITE HEART.     |17850.0   |3.39     |true             |true          |true       |
|536365   |SET 7 BABUSHKA NESTING BOXES       |17850.0   |7.65     |true             |tr

In [30]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

original_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

duplicate_window = Window.partitionBy(*original_columns).orderBy("InvoiceNo")

quality_df = quality_df.withColumn(
    "duplicate_number",
    row_number().over(duplicate_window)
)

quality_df = quality_df.withColumn(
    "unique_valid",
    col("duplicate_number") == 1
)

print(
    "Duplicate rows detected:",
    quality_df.filter(col("unique_valid") == False).count()
)

quality_df.filter(
    col("unique_valid") == False
).select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "unique_valid"
).show(10, truncate=False)

Duplicate rows detected: 5268
+---------+---------+-----------------------------------+--------+---------+------------+
|InvoiceNo|StockCode|Description                        |Quantity|UnitPrice|unique_valid|
+---------+---------+-----------------------------------+--------+---------+------------+
|536409   |22866    |HAND WARMER SCOTTY DOG DESIGN      |1       |2.1      |false       |
|536412   |21448    |12 DAISY PEGS IN WOOD BOX          |1       |1.65     |false       |
|536412   |21448    |12 DAISY PEGS IN WOOD BOX          |2       |1.65     |false       |
|536412   |21448    |12 DAISY PEGS IN WOOD BOX          |2       |1.65     |false       |
|536412   |21706    |FOLDING UMBRELLA RED/WHITE POLKADOT|1       |4.95     |false       |
|536412   |22273    |FELTCRAFT DOLL MOLLY               |1       |2.95     |false       |
|536412   |22327    |ROUND SNACK BOXES SET OF 4 SKULLS  |1       |2.95     |false       |
|536412   |22569    |FELTCRAFT CUSHION BUTTERFLY        |2       |3.75

In [31]:
from pyspark.sql.functions import when

quality_df = quality_df.withColumn(
    "quality_status",
    when(
        col("description_valid") &
        col("customer_valid") &
        col("price_valid") &
        col("unique_valid"),
        "PASS"
    ).otherwise("FAIL")
)

print("Quality Gate Results:")

quality_df.groupBy("quality_status").count().show()

Quality Gate Results:
+--------------+------+
|quality_status| count|
+--------------+------+
|          FAIL|140345|
|          PASS|401564|
+--------------+------+



In [32]:
trusted_df = quality_df.filter(col("quality_status") == "PASS")

quarantine_df = quality_df.filter(col("quality_status") == "FAIL")

print("Trusted/PASS records:", trusted_df.count())
print("Quarantine/FAIL records:", quarantine_df.count())

Trusted/PASS records: 401564
Quarantine/FAIL records: 140345


In [33]:
from pyspark.sql.functions import concat_ws, array, when, lit

quarantine_df = quarantine_df.withColumn(
    "failure_reason",
    concat_ws(
        ", ",
        array(
            when(~col("description_valid"), lit("Missing Description")),
            when(~col("customer_valid"), lit("Missing CustomerID")),
            when(~col("price_valid"), lit("Invalid UnitPrice")),
            when(~col("unique_valid"), lit("Duplicate Record"))
        )
    )
)

quarantine_df.select(
    "InvoiceNo",
    "Description",
    "CustomerID",
    "UnitPrice",
    "failure_reason"
).show(10, truncate=False)

+---------+-----------------------------------+----------+---------+----------------------------------------------------------+
|InvoiceNo|Description                        |CustomerID|UnitPrice|failure_reason                                            |
+---------+-----------------------------------+----------+---------+----------------------------------------------------------+
|536409   |HAND WARMER SCOTTY DOG DESIGN      |17908.0   |2.1      |Duplicate Record                                          |
|536412   |12 DAISY PEGS IN WOOD BOX          |17920.0   |1.65     |Duplicate Record                                          |
|536412   |12 DAISY PEGS IN WOOD BOX          |17920.0   |1.65     |Duplicate Record                                          |
|536412   |12 DAISY PEGS IN WOOD BOX          |17920.0   |1.65     |Duplicate Record                                          |
|536412   |FOLDING UMBRELLA RED/WHITE POLKADOT|17920.0   |4.95     |Duplicate Record                    

In [34]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

spark.stop()

builder = (
    SparkSession.builder
    .appName("EcommerceSalesDataPipeline")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark restarted with Delta Lake support!")
print("Spark version:", spark.version)

Spark restarted with Delta Lake support!
Spark version: 4.0.4


In [35]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/content/online_retail.csv")
)

print("Data ingested successfully!")
print("Total rows:", raw_df.count())

raw_df.show(5, truncate=False)

Data ingested successfully!
Total rows: 541909
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850.0   |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39  

In [36]:
from pyspark.sql.functions import col, trim, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Original dataset columns
original_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

# 1. Data Quality Checks
quality_df = (
    raw_df
    .withColumn(
        "description_valid",
        col("Description").isNotNull() & (trim(col("Description")) != "")
    )
    .withColumn(
        "customer_valid",
        col("CustomerID").isNotNull()
    )
    .withColumn(
        "price_valid",
        col("UnitPrice") > 0
    )
)

# 2. Uniqueness Check
duplicate_window = Window.partitionBy(*original_columns).orderBy("InvoiceNo")

quality_df = (
    quality_df
    .withColumn(
        "duplicate_number",
        row_number().over(duplicate_window)
    )
    .withColumn(
        "unique_valid",
        col("duplicate_number") == 1
    )
)

# 3. Quality Gate
quality_df = quality_df.withColumn(
    "quality_status",
    when(
        col("description_valid") &
        col("customer_valid") &
        col("price_valid") &
        col("unique_valid"),
        "PASS"
    ).otherwise("FAIL")
)

print("Quality Gate Results:")
quality_df.groupBy("quality_status").count().show()

Quality Gate Results:
+--------------+------+
|quality_status| count|
+--------------+------+
|          FAIL|140345|
|          PASS|401564|
+--------------+------+



In [37]:
from pyspark.sql.functions import concat_ws, array, lit

# PASS records
trusted_df = quality_df.filter(col("quality_status") == "PASS")

# FAIL records
quarantine_df = quality_df.filter(col("quality_status") == "FAIL")

# Add failure reason to quarantined records
quarantine_df = quarantine_df.withColumn(
    "failure_reason",
    concat_ws(
        ", ",
        array(
            when(~col("description_valid"), lit("Missing Description")),
            when(~col("customer_valid"), lit("Missing CustomerID")),
            when(~col("price_valid"), lit("Invalid UnitPrice")),
            when(~col("unique_valid"), lit("Duplicate Record"))
        )
    )
)

print("Trusted/PASS records:", trusted_df.count())
print("Quarantine/FAIL records:", quarantine_df.count())

print("\nSample quarantined records:")
quarantine_df.select(
    "InvoiceNo",
    "Description",
    "CustomerID",
    "UnitPrice",
    "failure_reason"
).show(5, truncate=False)

Trusted/PASS records: 401564
Quarantine/FAIL records: 140345

Sample quarantined records:
+---------+-----------------------------------+----------+---------+----------------+
|InvoiceNo|Description                        |CustomerID|UnitPrice|failure_reason  |
+---------+-----------------------------------+----------+---------+----------------+
|536409   |HAND WARMER SCOTTY DOG DESIGN      |17908.0   |2.1      |Duplicate Record|
|536412   |12 DAISY PEGS IN WOOD BOX          |17920.0   |1.65     |Duplicate Record|
|536412   |12 DAISY PEGS IN WOOD BOX          |17920.0   |1.65     |Duplicate Record|
|536412   |12 DAISY PEGS IN WOOD BOX          |17920.0   |1.65     |Duplicate Record|
|536412   |FOLDING UMBRELLA RED/WHITE POLKADOT|17920.0   |4.95     |Duplicate Record|
+---------+-----------------------------------+----------+---------+----------------+
only showing top 5 rows


In [38]:
delta_path = "/content/delta/trusted_sales"

(
    trusted_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(delta_path)
)

print("Trusted data successfully stored in Delta Lake!")

Trusted data successfully stored in Delta Lake!


In [39]:
trusted_delta_df = (
    spark.read
    .format("delta")
    .load("/content/delta/trusted_sales")
)

print("Trusted data loaded from Delta Lake successfully!")
print("Trusted records:", trusted_delta_df.count())

trusted_delta_df.show(5, truncate=False)

Trusted data loaded from Delta Lake successfully!
Trusted records: 401564
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+-----------------+--------------+-----------+----------------+------------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |description_valid|customer_valid|price_valid|duplicate_number|unique_valid|quality_status|
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+-----------------+--------------+-----------+----------------+------------+--------------+
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|true             |true          |true       |1               |true        |PASS          |
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-

In [40]:

from pyspark.sql.functions import col

analytics_df = (
    trusted_delta_df
    .filter(col("Quantity") > 0)
    .withColumn(
        "TotalSales",
        col("Quantity") * col("UnitPrice")
    )
)

print("Sales records ready for analytics:", analytics_df.count())

analytics_df.select(
    "InvoiceNo",
    "Description",
    "Quantity",
    "UnitPrice",
    "TotalSales",
    "Country"
).show(5, truncate=False)

Sales records ready for analytics: 392692
+---------+---------------------------------+--------+---------+------------------+--------------+
|InvoiceNo|Description                      |Quantity|UnitPrice|TotalSales        |Country       |
+---------+---------------------------------+--------+---------+------------------+--------------+
|536365   |GLASS STAR FROSTED T-LIGHT HOLDER|6       |4.25     |25.5              |United Kingdom|
|536365   |SET 7 BABUSHKA NESTING BOXES     |2       |7.65     |15.3              |United Kingdom|
|536365   |WHITE METAL LANTERN              |6       |3.39     |20.34             |United Kingdom|
|536365   |RED WOOLLY HOTTIE WHITE HEART.   |6       |3.39     |20.34             |United Kingdom|
|536366   |HAND WARMER UNION JACK           |6       |1.85     |11.100000000000001|United Kingdom|
+---------+---------------------------------+--------+---------+------------------+--------------+
only showing top 5 rows


In [41]:
from pyspark.sql.functions import sum, round

sales_by_country = (
    analytics_df
    .groupBy("Country")
    .agg(
        round(sum("TotalSales"), 2).alias("TotalSales")
    )
    .orderBy(col("TotalSales").desc())
)

print("Top 10 Countries by Total Sales:")
sales_by_country.show(10, truncate=False)

Top 10 Countries by Total Sales:
+--------------+----------+
|Country       |TotalSales|
+--------------+----------+
|United Kingdom|7285024.64|
|Netherlands   |285446.34 |
|EIRE          |265262.46 |
|Germany       |228678.4  |
|France        |208934.31 |
|Australia     |138453.81 |
|Spain         |61558.56  |
|Switzerland   |56443.95  |
|Belgium       |41196.34  |
|Sweden        |38367.83  |
+--------------+----------+
only showing top 10 rows


In [42]:
from pyspark.sql.functions import sum

top_products = (
    analytics_df
    .groupBy("StockCode", "Description")
    .agg(
        sum("Quantity").alias("TotalQuantitySold")
    )
    .orderBy(col("TotalQuantitySold").desc())
)

print("Top 10 Products by Quantity Sold:")
top_products.show(10, truncate=False)

Top 10 Products by Quantity Sold:
+---------+----------------------------------+-----------------+
|StockCode|Description                       |TotalQuantitySold|
+---------+----------------------------------+-----------------+
|23843    |PAPER CRAFT , LITTLE BIRDIE       |80995            |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |77916            |
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS |54319            |
|85099B   |JUMBO BAG RED RETROSPOT           |46078            |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|36706            |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |35263            |
|21212    |PACK OF 72 RETROSPOT CAKE CASES   |33670            |
|22197    |POPCORN HOLDER                    |30919            |
|23084    |RABBIT NIGHT LIGHT                |27153            |
|22492    |MINI PAINT SET VINTAGE            |26076            |
+---------+----------------------------------+-----------------+
only showing top 10 rows


In [43]:
from pyspark.sql.functions import date_format, sum, round, col

monthly_sales = (
    analytics_df
    .withColumn(
        "Month",
        date_format(col("InvoiceDate"), "yyyy-MM")
    )
    .groupBy("Month")
    .agg(
        round(sum("TotalSales"), 2).alias("TotalSales")
    )
    .orderBy("Month")
)

print("Monthly Sales Trend:")
monthly_sales.show(20, truncate=False)

Monthly Sales Trend:
+-------+----------+
|Month  |TotalSales|
+-------+----------+
|2010-12|570422.73 |
|2011-01|568101.31 |
|2011-02|446084.92 |
|2011-03|594081.76 |
|2011-04|468374.33 |
|2011-05|677355.15 |
|2011-06|660046.05 |
|2011-07|598962.9  |
|2011-08|644051.04 |
|2011-09|950690.2  |
|2011-10|1035642.45|
|2011-11|1156205.61|
|2011-12|517190.44 |
+-------+----------+



In [44]:
quarantine_path = "/content/quarantine/failed_records"

(
    quarantine_df
    .write
    .mode("overwrite")
    .parquet(quarantine_path)
)

# Verify the quarantined data
quarantine_check = spark.read.parquet(quarantine_path)

print("Quarantine data stored successfully!")
print("Quarantined records:", quarantine_check.count())

Quarantine data stored successfully!
Quarantined records: 140345
